In [ ]:
!python -m pip install -U python-woc pandas matplotlib
# please clear the output of this cell in your notebook before checking it in

In [3]:
from woc.remote import WocMapsRemote
from tqdm import tqdm
import pandas as pd
import time
# creates the client
woc = WocMapsRemote( base_url="https://worldofcode.org/api/")
# if you got an API key
# woc = WocMapsRemote( base_url="https://worldofcode.org/api/", api_key="woc-XXXXXX-YYYYYY" )

# Now for each of the ten projects you were assigned get commits, e.g.
projects = [
    "patwie_cppnumericalsolvers",
    "simplenlg_simplenlg",
    "ntustison_crosslong",
    "stan-dev_projpred",
    "whizzml_examples",
    "afeinstein20_eleanor",
    "smtorg_smt",
    "algebraicjulia_catlab.jl",
    "microprediction_timemachines",
    "sftcd_surveys",
]
list_df_commits = []
  # Convert GitHub repository names to WoC V2412 project names
for prj in projects:
    try:
        commits = woc.get_values('p2c', prj)
    except Exception as e:
        print(prj, "FAILED:", e)
        continue
    d = pd.DataFrame(commits, columns=["sha1"])
    d['project'] = prj
    list_df_commits.append(d)
    print(f"{prj}: {len(d)} commits")
    time.sleep(1)

df = pd.concat(list_df_commits, ignore_index=True)
df.to_csv('df_commits.csv', index=False)
print(len(df), "rows,", df['sha1'].nunique(), "unique commits")

patwie_cppnumericalsolvers: 335 commits
simplenlg_simplenlg: 136 commits
ntustison_crosslong: 311 commits
stan-dev_projpred: 6807 commits
whizzml_examples: 527 commits
afeinstein20_eleanor: 1667 commits
smtorg_smt: 3823 commits
algebraicjulia_catlab.jl: 4645 commits
microprediction_timemachines: 1441 commits
sftcd_surveys: 392 commits
20084 rows, 20084 unique commits


In [4]:
# If the number of commits its not very large, you can try to get them all at the same time,
# The max batch size is 10
import time

shas = df['sha1'].tolist()
chunks = [shas[x:x+10] for x in range(0, len(shas), 10)]
commit_data = []

for chunk in tqdm(chunks):
    res, err = woc.get_values_many('commit.tch', chunk)
    res = {k: v[0] for k, v in res.items()}
    time.sleep(1)
    if err:
        print('Got Errors', err)
    for commit_sha, commit in res.items():
        commit_data.append({
            'commit': commit_sha,
            'author': commit[2][0],
            'author_time': int(commit[2][1]),
            'message': commit[4],
        })

 22%|████████████████▏                                                       | 450/2009 [07:50<26:58,  1.04s/it]

Got Errors {'8c185e98221ed98fff3e1e7440e1def090df5d1c': 'Key 8c185e98221ed98fff3e1e7440e1def090df5d1c not found in /da5_fast/All.sha1c/commit_12.tch'}


 24%|█████████████████                                                       | 477/2009 [08:18<26:30,  1.04s/it]

Got Errors {'961a77e62b1191312946b995111e60528c318104': 'Key 961a77e62b1191312946b995111e60528c318104 not found in /da5_fast/All.sha1c/commit_22.tch'}


 37%|██████████████████████████▉                                             | 751/2009 [13:02<21:43,  1.04s/it]

...

 57%|████████████████████████████████████████▌                              | 1148/2009 [19:58<15:19,  1.07s/it]

Got Errors {'70839453dd15c4f2b0b11efb785a33d44310653c': 'Key 70839453dd15c4f2b0b11efb785a33d44310653c not found in /da5_fast/All.sha1c/commit_112.tch'}


 62%|████████████████████████████████████████████▎                          | 1253/2009 [21:48<13:04,  1.04s/it]

Got Errors {'b7ee7bd6d5e22533a0dd78279c3418fb05e71c60': 'Key b7ee7bd6d5e22533a0dd78279c3418fb05e71c60 not found in /da5_fast/All.sha1c/commit_55.tch'}


100%|███████████████████████████████████████████████████████████████████████| 2009/2009 [34:58<00:00,  1.04s/it]


In [5]:
# what does the first record looks like?
# commit  parents
v = commit_data[0]
# commit tree, [parent{s}], [author, unixtime, tz ], [ committer, unixtime, tz ], Commit_message ]
print(v)

{'commit': '0108c3e639f6ae3a97136d6d2a04d5ea045b4697', 'author': 'patwie <mail@patwie.com>', 'author_time': 1740597536, 'message': 'Introduce differential template argument\n\nWe want pure virtual classes. Making the existence of a gradient, and\nHessian member method as compile-time check. This removes the finite\ndifferences as a default gradient, Hessian, which should be anyway an\nexplicit choice.\n'}


In [12]:
df_commit_data = pd.DataFrame(commit_data).merge(df, left_on='commit', right_on='sha1')

dfinf = pd.DataFrame({
    'project_wocid': df_commit_data['project'],
    'commit_sha1': df_commit_data['commit'],
    'author': df_commit_data['author'],
    'time': df_commit_data['author_time'],
    'commit_message': df_commit_data['message'],
})
dfinf.to_csv('ncash3_project_summary.csv', index=False, sep=';', header=False)

stats = df_commit_data.groupby('project').agg(
    n_commits=('commit', 'nunique'),
    n_authors=('author', 'nunique'),
    min_time=('author_time', 'min'),
    max_time=('author_time', 'max'),
)
stats['min_time'] = pd.to_datetime(stats['min_time'], unit='s')
stats['max_time'] = pd.to_datetime(stats['max_time'], unit='s')
stats

,n_commits,n_authors,min_time,max_time
project,,,,
afeinstein20_eleanor,1665,40,2018-07-24 22:02:15,2025-08-26 04:31:09
algebraicjulia_catlab.jl,4645,88,2017-01-24 07:58:34,2026-04-17 01:09:29
microprediction_timemachines,1441,14,2021-01-04 15:38:42,2024-10-21 14:44:48
ntustison_crosslong,311,23,2016-06-29 21:39:39,2025-07-16 18:55:21
patwie_cppnumericalsolvers,335,40,2014-05-29 15:45:13,2026-01-08 11:01:06
sftcd_surveys,392,11,2017-09-15 12:21:00,2025-08-14 21:37:16
simplenlg_simplenlg,136,23,2014-08-20 21:37:08,2024-12-06 12:07:07
smtorg_smt,3821,98,2016-11-08 23:00:32,2025-10-31 14:25:24
stan-dev_projpred,6804,34,2016-06-16 11:58:02,2025-11-01 05:08:09


In [13]:
# check if it has the right content
dfinf.head(1)

,project_wocid,commit_sha1,author,time,commit_message
0,patwie_cppnumericalsolvers,0108c3e639f6ae3a97136d6d2a04d5ea045b4697,patwie <mail@patwie.com>,1740597536,Introduce differential template argument\n\nWe...


In [14]:
!head -c 300 ncash3_project_summary.csv

patwie_cppnumericalsolvers;0108c3e639f6ae3a97136d6d2a04d5ea045b4697;patwie <mail@patwie.com>;1740597536;"Introduce differential template argument

We want pure virtual classes. Making the existence of a gradient, and
Hessian member method as compile-time check. This removes the finite
differences as

# Make sure you check in to your fork not just the notebook but also the csv files!!!

# Don't forget to add requested data from github and this notebook
### For each of the 10 projects go to their github repo and get the number of stars, number of forks, and the last commit date
### Report (in your notebook) the number of commits, the number of authors, and max and min time for each project based on WoC commits and also add the info you obtained from github

In [16]:
import requests, time

projects_gh = {
    "patwie_cppnumericalsolvers": "PatWie/CppNumericalSolvers",
    "simplenlg_simplenlg": "simplenlg/simplenlg",
    "ntustison_crosslong": "ntustison/CrossLong",
    "stan-dev_projpred": "stan-dev/projpred",
    "whizzml_examples": "whizzml/examples",
    "afeinstein20_eleanor": "afeinstein20/eleanor",
    "smtorg_smt": "SMTorg/smt",
    "algebraicjulia_catlab.jl": "AlgebraicJulia/Catlab.jl",
    "microprediction_timemachines": "microprediction/timemachines",
    "sftcd_surveys": "sftcd/surveys",
}

gh = {}
for wocid, repo in projects_gh.items():
    info = requests.get(f"https://api.github.com/repos/{repo}").json()
    c = requests.get(f"https://api.github.com/repos/{repo}/commits", params={"per_page": 1}).json()
    gh[wocid] = {
        "stars": info["stargazers_count"],
        "forks": info["forks_count"],
        "last_commit": c[0]["commit"]["committer"]["date"],
    }
    time.sleep(1)

pd.DataFrame(gh).T

,stars,forks,last_commit
patwie_cppnumericalsolvers,978,205,2026-07-24T14:45:03Z
simplenlg_simplenlg,832,185,2024-12-06T12:07:07Z
ntustison_crosslong,0,0,2026-08-11T20:35:10Z
stan-dev_projpred,114,31,2026-09-26T10:10:01Z
whizzml_examples,24,42,2025-05-23T17:56:00Z
afeinstein20_eleanor,101,43,2026-07-17T21:38:37Z
smtorg_smt,916,231,2026-09-17T08:03:24Z
algebraicjulia_catlab.jl,727,73,2026-07-02T15:25:46Z
microprediction_timemachines,443,54,2026-09-24T16:28:02Z
sftcd_surveys,5,5,2026-07-03T19:23:22Z


## Summary

| Project | Num of stars | Num of forks | Last commit date | Num of commits | Num of authors | Min time | Max time |
|---|---|---|---|---|---|---|---|
| patwie_cppnumericalsolvers | 978 | 205 | 2026-07-24 | 335 | 40 | 2014-05-29 15:45:13 | 2026-01-08 11:01:06 |
| simplenlg_simplenlg | 832 | 185 | 2024-12-06 | 136 | 23 | 2014-08-20 21:37:08 | 2024-12-06 12:07:07 |
| ntustison_crosslong | 0 | 0 | 2026-08-11 | 311 | 23 | 2016-06-29 21:39:39 | 2025-07-16 18:55:21 |
| stan-dev_projpred | 114 | 31 | 2026-09-26 | 6804 | 34 | 2016-06-16 11:58:02 | 2025-11-01 05:08:09 |
| whizzml_examples | 24 | 42 | 2025-05-23 | 525 | 27 | 2016-05-14 19:29:21 | 2025-05-23 19:56:13 |
| afeinstein20_eleanor | 101 | 43 | 2026-07-17 | 1665 | 40 | 2018-07-24 22:02:15 | 2025-08-26 04:31:09 |
| smtorg_smt | 916 | 231 | 2026-09-17 | 3821 | 98 | 2016-11-08 23:00:32 | 2025-10-31 14:25:24 |
| algebraicjulia_catlab.jl | 727 | 73 | 2026-07-02 | 4645 | 88 | 2017-01-24 07:58:34 | 2026-04-17 01:09:29 |
| microprediction_timemachines | 443 | 54 | 2026-09-24 | 1441 | 14 | 2021-01-04 15:38:42 | 2024-10-21 14:44:48 |
| sftcd_surveys | 5 | 5 | 2026-07-03 | 392 | 11 | 2017-09-15 12:21:00 | 2025-08-14 21:37:16 |
